In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from accelerate import Accelerator
from PromptTemplate import PromptTemplate
from HuggingFaceModel import HuggingFaceModel
from TrainStrategy import TrainStrategy
from constant import *
from LlmSatdOutputLabelConverter import LlmSatdOutputLabelConverter

/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
prompt_template = PromptTemplate(
    name="Manually Crafted",
    definition="You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior, provide instructions, or reference external issues (e.g., JIRA ID) are not SATD unless there is additional information indicating the need for future improvement.",
    instruction="Think step by step and assign the label of **SATD** or **Not-SATD** for each given test code comment.",
    n_shot_template="Comment: {{ text }}",
    n_shot_answer_template="""{% if cot -%}
        Answer: {{ cot }} The answer is **{{ label }}**.
        {% endif -%}""",
    line_m_before=3,
    line_n_after=3
)
output_label_converter = LlmSatdOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

In [5]:
for model_name in ['google/flan-t5-small']:
    flan_t5_detection_model = HuggingFaceModel('detect', model_name, output_label_converter)
    flan_t5_detection_model.fit(detect_n_shot_dataset)
    flan_t5_detection_model.predict(detect_test_dataset.select(range(10)), DETECT_DATASET_NAME,  prompt_template,TrainStrategy.N_SHOT_TOP,2, verbose=True)


detect with flan-t5-small
Prompt:
 You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior, provide instructions, or reference external issues (e.g., JIRA ID) are not SATD unless there is additional information indicating the need for future improvement. Think step by step and assign the label of **SATD** or **Not-SATD** for each given test code comment.
Comment: //hacky workaround using the toString method to avoid mocki

# Flan T5 Models

In [ ]:
for model_name in ['google/flan-t5-small', 'google/flan-t5-base', 'google/flan-t5-large', 'google/flan-t5-xl', 'google/flan-t5-xxl']:
    flan_t5_detection_model = HuggingFaceModel('detect', model_name, output_label_converter)
    flan_t5_detection_model.fit(detect_n_shot_dataset)
    flan_t5_detection_model.predict(detect_test_dataset, DETECT_DATASET_NAME,  prompt_template,TrainStrategy.N_SHOT_TOP,0, verbose=False)
